# Day 1 — Applied Lab: Building a Resolution Agent
### Agentic Customer Experience Specialisation · Post-lunch session (4h)

**What we're building today:** a chat resolution agent that answers customer questions using a
real knowledge base (not general knowledge), speaks in a defined persona rather than a generic
voice, takes one real action safely, and contains or escalates cleanly depending on whether it
genuinely can help. By the end, the agent either resolves a real customer query end-to-end or
hands it off with enough context that a human doesn't have to start over.

**How the six post-lunch topics map onto this notebook:**

| # | Topic | Where |
|---|---|---|
| 1 | Agent architecture | Architecture section, right after Setup |
| 2 | Resolution flows | Steps 1–4 (knowledge base → retrieval → grounding) |
| 3 | Persona & tone | Step 5 |
| 4 | Containment & escalation | Step 6 |
| 5 | Actions & confirmations | Lab 2 |
| 6 | Instrumentation | Lab 3 |

**Demo lane:** Insurance. The pattern — retrieval, grounding, escalation, confirmed actions,
instrumentation — is the same regardless of domain; only the knowledge base and the action tool
change. Banking, Retail, and Telecom variants are in the appendix at the end.

**How this notebook is organized:** each code cell is preceded by an explanation of what it does
and, more importantly, *why it's built this way* — what would go wrong without that piece. A few
markers recur throughout:

| Marker | Meaning |
|---|---|
| ▶ Run this | A cell you execute and observe the output of |
| ✅ Checkpoint | A natural save point — the agent is in a working, shippable state |
| 🔍 Try it yourself | A prompt to test with your own input, not just the example given |

There's no marker for "scripted failure" — a couple of cells are built to demonstrate a specific
failure mode on purpose (an ungrounded answer, a rejected duplicate action). The explanation before
each one says so directly, and says what to expect.


## Setup

**Prerequisites (Install these following beforehand):**
1. Node.js + npm
2. The Claude Code CLI, which the Agent SDK uses as its runtime: `npm install -g @anthropic-ai/claude-code`
3. Python 3.10+, then `pip install claude-agent-sdk anthropic`
4. `export ANTHROPIC_API_KEY=your-key`

**Why the CLI matters:** the Claude Agent SDK doesn't talk to the Anthropic API directly — it
shells out to the Claude Code CLI as its runtime. That's not incidental scaffolding; skip step 2
and the very next cell raises `CLINotFoundError`. It's the single most common setup failure, so
it's worth confirming before anything else.

**One more prerequisite that's easy to miss:** every code cell below uses `await` directly at the
top level (not inside an `async def`). That needs a reasonably current Jupyter/IPython kernel with
autoawait support. On an older kernel, or if a cell gets copied into a plain `.py` script, this
fails immediately with `SyntaxError: 'await' outside function`. Confirm your kernel version now,
not four cells in.

▶ **Run this** — it imports the SDK and asserts both the API key and the CLI are in place. If it
prints `Environment ready.`, you're set for the rest of the notebook.


In [2]:
# Setup — run this first.
import os, json, time, shutil

# claude_agent_sdk is a thin Python layer over the Claude Code CLI, not a direct HTTP
# client — everything below eventually shells out to the `claude` binary on PATH.
#   tool                 -> decorator: turns a plain async function into a callable tool
#   create_sdk_mcp_server -> bundles one or more @tool functions into an in-process MCP server
#   ClaudeAgentOptions    -> config only (persona, tools, guardrails) — inert until used
#   ClaudeSDKClient       -> the actual session object that drives a conversation
from claude_agent_sdk import tool, create_sdk_mcp_server, ClaudeAgentOptions, ClaudeSDKClient

# Both checks below are pure Python / filesystem checks — nothing has talked to
# Claude yet. They exist to fail fast with a readable message instead of a raw
# CLINotFoundError three cells from now.
assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY before continuing"
assert shutil.which("claude"), (
    # shutil.which = "is `claude` on PATH", not "does it run"
    "Claude Code CLI not found on PATH — run: npm install -g @anthropic-ai/claude-code"
)
print("Environment ready.")


Environment ready.


In [3]:
import asyncio
import sys


def run_sdk(coro):
    """Drive a claude_agent_sdk coroutine to completion on a freshly-created
    Proactor event loop, in whatever thread this runs on (see call sites below,
    which schedule it via asyncio.to_thread).

    Why: claude_agent_sdk spawns the Claude Code CLI as a subprocess. On
    Windows, only ProactorEventLoop supports asyncio subprocesses --
    SelectorEventLoop raises a bare NotImplementedError. Some Jupyter kernels
    end up running on a SelectorEventLoop, and by the time a notebook cell
    runs, that loop already exists -- so this builds a fresh loop of the
    right type per-call via asyncio.Runner(loop_factory=...) instead of
    mutating the (deprecated, 3.16-removal) global event loop policy.
    """
    loop_factory = asyncio.ProactorEventLoop if sys.platform == "win32" else None
    with asyncio.Runner(loop_factory=loop_factory) as runner:
        return runner.run(coro)


print("run_sdk() helper ready (Windows/Jupyter subprocess workaround).")


run_sdk() helper ready (Windows/Jupyter subprocess workaround).


## Architecture — what's actually running when you call `ask()`

Before wiring up any tool, it's worth seeing the whole shape once — every piece below gets built
live over the next two hours, but knowing where each one fits *before* you write it makes each
step a slot-in instead of a surprise.

```
 customer message
        │
        ▼
┌────────────────────────┐   .query(text) / .receive_response()
│   ClaudeSDKClient        │◄──────────────────────────────────────────┐
│   — the session. Keeps   │                                            │
│   turns 1..N in context  │        ClaudeAgentOptions (config, not code)
│   (Lab 2's preview→      │        • system_prompt      — persona + rules
│   confirm needs this)    │        • mcp_servers        — which tool servers are wired in
└────────────┬─────────────┘        • allowed_tools      — explicit per-tool allowlist
             │                      • disallowed_tools   — hard lockdown (BUILTIN_LOCKDOWN)
             │ spawns / drives
             ▼
┌────────────────────────┐
│   Claude Code CLI         │   the actual execution runtime. The SDK is a wrapper around this
│   (subprocess)             │   process, not a direct HTTP client to the API — that's why a
└────────────┬─────────────┘   missing CLI install fails Setup, not just "some feature."
             │
             │ each turn: the model decides "answer now" or "call a tool"
             ▼
┌────────────────────────┐   tool_use ──────────────────────────────┐
│   claude (the model)      │                                          ▼
└────────────┬─────────────┘                          ┌─────────────────────────────┐
             │ tool_result                              │  create_sdk_mcp_server        │
             │◄─────────────────────────────────────────┤  (in-process MCP server)      │
             ▼                                          │  runs your @tool-decorated     │
   answer, or another tool call, or done                │  Python function directly      │
                                                          └─────────────────────────────┘
```

**This is the agentic CX loop from this morning's session, made concrete:** perceive (the
customer's message) → reason (does this need a tool?) → act (call `kb_search`, `file_claim`, …) →
observe (the tool's result) → respond or loop again. Every lab this afternoon is that same loop
with one more tool added to the "act" step.

**Where each piece shows up in this notebook, and the docs for going deeper than today's lab:**

| Piece | Role today | Docs |
|---|---|---|
| `ClaudeAgentOptions` | Declares persona, which tools exist, which are allowed — config, not control flow | https://docs.claude.com/en/api/agent-sdk/python |
| `ClaudeSDKClient` | Drives a session; required (not `ask()`) whenever a later turn depends on an earlier one — Lab 2's confirm step | https://docs.claude.com/en/api/agent-sdk/python |
| `@tool` + `create_sdk_mcp_server` | Turns a plain Python function into a callable tool, served in-process | https://docs.claude.com/en/api/agent-sdk/python |
| Claude Code CLI | The runtime the SDK shells out to — not optional scaffolding | https://docs.claude.com/en/docs/claude-code/overview |
| MCP (Model Context Protocol) | The open protocol tool-calling runs on. Today's server is in-process; production MCP servers are frequently separate processes/services speaking the same protocol over the network | https://modelcontextprotocol.io/introduction |

*(Doc links point at the current Claude docs site — worth a quick click-through before presenting,
docs structure shifts over time.)*

**The four `ClaudeAgentOptions` fields above are the only ones this lab uses — not the only ones
that exist.** Also available, out of scope for today but worth naming so "wait, can it also do
X?" has an answer: `permission_mode` (interactive approval instead of a static allowlist),
`hooks` (run code on specific SDK lifecycle events), `max_turns` (hard cap on a runaway
conversation), `model` (pin a specific Claude model version), `cwd` / `setting_sources` (filesystem
and settings scoping — mostly relevant when the SDK is used for coding-agent tasks, not CX ones).

**Reading the diagram as one guardrail statement:** everything the agent is capable of doing is
*declared* in `ClaudeAgentOptions` before the conversation starts — not decided by the model
mid-conversation. `disallowed_tools` in Step 2 is the first concrete example: the agent isn't
capable of touching the filesystem not because it chooses not to, but because that capability was
never wired in.

Nothing to run yet — Step 1 starts building the pieces on the left of this diagram.


## Step 1 — The knowledge base

This is the source of truth the agent will be required to ground its answers in: four clauses from
an auto insurance policy.

```
POL-4.2  Comprehensive coverage — rental reimbursement exclusion
POL-4.3  Collision coverage — scope (insured vehicle only)
POL-9.1  Rider R-12 — rental reimbursement, if purchased separately
POL-2.5  Claim requirements — incident report + repair estimate
```

**Why `score()` is a keyword-overlap function, not real embeddings:** in production this would be
replaced by a real embedding model (Voyage AI is Anthropic's recommended provider) plus a vector
store. Here it's deliberately simplified so the lab runs with zero extra API keys or infrastructure.
The thing to focus on isn't the scoring math — it's the **retrieval pattern**: chunk by clause (not
by arbitrary character count), and keep the source id attached to every chunk, all the way through
to the final answer.

**Why POL-4.2 and POL-9.1 matter together:** read in isolation, they look like they contradict each
other — one says rental cars aren't covered, the other describes rental reimbursement. They don't
actually conflict: 4.2 is the base policy's exclusion, 9.1 is an optional rider that overrides it if
purchased. Getting this question right requires retrieving *both* clauses, not just the
best-matching one. That's built in on purpose — it's the whole reason this specific question is
used as the running example.

▶ **Run this** and check the printed ranking for the sample query — both POL-4.2 and POL-9.1 should
land in the top 3 results. If they don't, the live-code demo two steps from now won't reproduce
correctly.


In [4]:
policy_chunks = [
    {"id": "POL-4.2", "text": "Comprehensive coverage does not include rental vehicle "
     "reimbursement unless Rider R-12 (Rental Reimbursement) has been purchased separately."},
    {"id": "POL-4.3", "text": "Collision coverage applies to damages resulting from an accident "
     "involving the insured vehicle and does not extend to third-party rental vehicles."},
    {"id": "POL-9.1", "text": "Rider R-12 provides up to INR 1,500/day for rental vehicle costs, "
     "capped at 30 days, while the insured vehicle is under repair due to a covered claim."},
    {"id": "POL-2.5", "text": "A covered claim requires an incident report filed within 7 days "
     "of the event and, for collision claims, a repair estimate from an approved garage."},
]

def score(query: str, text: str) -> float:
    # Jaccard-style overlap: (shared words) / (words in the query). Asymmetric on purpose —
    # a short query fully contained in a long clause still scores 1.0. This is the whole
    # stand-in for a retriever; swap it for an embedding model + vector store in production,
    # everything downstream (search(), the kb_search tool) stays the same shape.
    q = set(query.lower().split())
    t = set(text.lower().split())
    return len(q & t) / max(len(q), 1)

def search(query: str, top_k: int = 3):
    # O(n log n) full sort over 4 chunks — fine here, would not scale past a few thousand;
    # a real retriever uses an index (ANN/inverted index) instead of scoring every document.
    ranked = sorted(policy_chunks, key=lambda c: score(query, c["text"]), reverse=True)
    return ranked[:top_k]

# Quick sanity check — run this so trainees SEE retrieval before they see the agent use it.
# Both POL-4.2 and POL-9.1 must land in the top 3, or Step 4's grounded demo won't reproduce.
for r in search("does my policy cover a rental car after an accident"):
    print(r["id"], "-", r["text"])


POL-4.3 - Collision coverage applies to damages resulting from an accident involving the insured vehicle and does not extend to third-party rental vehicles.
POL-4.2 - Comprehensive coverage does not include rental vehicle reimbursement unless Rider R-12 (Rental Reimbursement) has been purchased separately.
POL-9.1 - Rider R-12 provides up to INR 1,500/day for rental vehicle costs, capped at 30 days, while the insured vehicle is under repair due to a covered claim.


## Step 2 — The retrieval tool

This wraps `search()` in a tool the agent can call, and registers it on an in-process MCP server.

**Why the tool's description matters as much as its code:** the description says *"Always call
this before answering any question about what is or isn't covered."* That sentence is doing real
work — it's the only thing that currently pushes the model toward retrieving instead of answering
from general knowledge. In the next section we'll see what happens when a tool exists but nothing
actually requires the model to use it.

**Two details here are easy to get wrong, and both fail silently rather than raising a clear
error:**
- `mcp_servers` on `ClaudeAgentOptions` takes a **dict** keyed by server name (`{"cx_tools": ...}`),
  not a list.
- Every tool the agent should be able to use without a permission prompt must be named exactly in
  `allowed_tools`, in the form `mcp__<server_name>__<tool_name>` — e.g. `mcp__cx_tools__kb_search`.
  Mismatch the server name between the two and the tool simply never gets called — there's no
  exception, the agent just doesn't have access.

**Why `BUILTIN_LOCKDOWN` exists:** this is a customer-facing CX agent, not a coding agent, so
Claude Code's own built-in tools (Bash, file edits, etc.) are explicitly blocked. Restricting an
agent to only the domain tools it needs is a guardrail — and guardrails aren't a topic reserved for
later in the course; they belong in the very first agent built.

▶ **Run this** and confirm `kb_search tool registered.` prints.


In [5]:
@tool(
    "kb_search",                                      # tool name — must match the mcp__<server>__<name>
    "Search the insurance policy knowledge base for clauses relevant to a customer's question. "
    "Always call this before answering any question about what is or isn't covered.",
    {"query": str},                                    # input schema: one required string arg
)
async def kb_search(args):
    # `args` is a plain dict matching the schema above — the SDK validates/parses the
    # model's tool-call JSON into this before your function ever runs.
    results = search(args["query"])
    formatted = "\n".join(f"[{r['id']}] {r['text']}" for r in results)
    # Every tool must return this exact shape — {"content": [{"type": "text", "text": ...}]} —
    # it's the MCP tool-result contract, not a claude_agent_sdk-specific convention.
    return {"content": [{"type": "text", "text": formatted}]}

# @tool wraps kb_search in an SdkMcpTool object (name + description + schema + handler) —
# the plain function is no longer directly callable after this; see the offline-check
# cells later in the notebook for the `.handler(...)` workaround.
cx_tools_server = create_sdk_mcp_server(
    name="cx_tools",
    version="1.0.0",
    tools=[kb_search],
)
# create_sdk_mcp_server runs the tool IN-PROCESS: no subprocess, no network hop for the
# tool call itself. The Claude Code CLI subprocess still mediates the model conversation —
# only the tool execution stays local. Production MCP servers are frequently separate
# processes/services; the calling convention looks identical either way.

# Block the built-in Claude Code tools (Bash, file edits, etc.) — this is a CX agent,
# it should only ever have the domain tools we explicitly give it. Worth calling out:
# guardrails aren't only a Day 4 topic, they start on day one.
BUILTIN_LOCKDOWN = ["Bash", "Read", "Write", "Edit", "Glob", "Grep"]

async def ask(options, question):
    """One-shot helper for single-turn tests. For multi-turn flows (Lab 2's
    preview -> confirm), use ClaudeSDKClient directly across two client.query() calls
    instead, so the session retains context."""
    # Opens a fresh session, sends one message, prints every message the model/tools
    # produce, then closes the session — nothing here survives past this function call.
    # That's fine for single-turn tests; Lab 2 needs the client kept open across turns.
    async with ClaudeSDKClient(options=options) as client:
        await client.query(question)
        async for message in client.receive_response():
            content = getattr(message, "content", None)
            if content:
                for block in content:
                    if hasattr(block, "text"):
                        print(f"\U0001F4AC {block.text}")
                    elif hasattr(block, "name"):
                        print(f"\U0001F527 called {block.name}({block.input})")

print("kb_search tool registered.")


kb_search tool registered.


## Step 3 — What happens without a grounding instruction

The next cell uses `weak_options`: the `kb_search` tool is available, but the system prompt is just
*"You are a helpful insurance customer service assistant."* — nothing tells the model it's required
to use the tool.

**What to expect, and why:** ask *"My car's in the shop after an accident — does my policy cover a
rental car?"* and the model will typically answer from its general insurance knowledge instead of
calling `kb_search` at all. The answer will sound plausible and confident — and be wrong for this
specific policy, which excludes rental cars unless Rider R-12 was purchased separately.

This is a deliberately engineered failure, not a bug — it's the clearest possible demonstration of
a real risk: an LLM with a tool available doesn't automatically use it. Absent an explicit
instruction, the model defaults to answering from training data, which is exactly the hallucination
risk that matters most in a CX context — a confident, wrong, uncited answer is worse than the
customer getting no answer at all.

🔍 **Try it yourself** with the exact question above before changing anything — this failure needs
to reproduce reliably for the next section to land.


In [6]:
weak_options = ClaudeAgentOptions(
    # No instruction to use kb_search — the tool is available but never mandated.
    # This is the actual bug under test: a capable tool sitting unused because
    # nothing in the prompt requires it.
    system_prompt="You are a helpful insurance customer service assistant.",
    mcp_servers={"cx_tools": cx_tools_server},          # dict keyed by server NAME, not a list
    allowed_tools=["mcp__cx_tools__kb_search"],         # must be mcp__<server_name>__<tool_name>
    disallowed_tools=BUILTIN_LOCKDOWN,
)

# Nothing runs until this call — ClaudeAgentOptions above was pure config.
await asyncio.to_thread(run_sdk, ask(weak_options, "My car's in the shop after an accident — does my policy cover a rental car?"))


💬 Depends on policy. Key points:

- **Rental reimbursement coverage** — separate add-on, not automatic. Check declarations page for it.
- No rental coverage listed → no rental pay, unless other driver at fault (their liability may cover it).
- If have coverage: daily $ cap + max days apply (e.g. $30/day, 30 days max).
- Collision/comprehensive claim (your fault or unknown) → rental reimbursement kicks in only if you bought it.

Next step: check declarations page for "rental reimbursement" or "transportation expenses" line. Or give me policy number / info, pull details.


## Step 4 — Fixing the grounding

`grounded_options` uses the same tool, but a system prompt that closes the gap from the previous
step. Read it clause by clause:

- *"For ANY question about coverage... you MUST call kb_search first"* — removes the option to
  skip retrieval.
- *"answer only from the returned clauses"* — stops the model blending in outside knowledge even
  after it retrieves.
- *"Always cite the clause id(s) you used"* — makes the answer checkable rather than just
  confident-sounding.
- *"If the retrieved clauses don't fully answer the question, say so — never fill gaps from general
  knowledge"* — this is the seed of the escalation logic added in the next step.

Run the identical question from the previous cell against `grounded_options`. Expect the agent to
call `kb_search`, retrieve both POL-4.2 and POL-9.1, and answer correctly, citing both.

**Why the citation matters more than the answer:** a citation that's present but *wrong* is
arguably worse than no citation at all — it looks trustworthy and isn't. When checking this output,
verify two separate things: a citation is present, and it's the *correct* clause, not just *a*
clause that happened to be retrieved.

🔍 **Try it yourself:** re-run the same question and confirm both conditions above hold.


In [7]:
grounded_options = ClaudeAgentOptions(
    system_prompt=(
        "You are an insurance customer service assistant. "
        # Removes the option to skip retrieval — the failure mode from the previous cell.
        "For ANY question about coverage, exclusions, or policy terms, you MUST call kb_search "
        # Stops the model blending in outside knowledge even after it retrieves.
        "first and answer only from the returned clauses. Always cite the clause id(s) you used, "
        # Makes the answer checkable instead of just confident-sounding.
        "in the form [POL-x.x]. If the retrieved clauses don't fully answer the question, say so — "
        # Seeds the escalation logic Step 6 adds later.
        "never fill gaps from general knowledge."
    ),
    mcp_servers={"cx_tools": cx_tools_server},
    allowed_tools=["mcp__cx_tools__kb_search"],
    disallowed_tools=BUILTIN_LOCKDOWN,
)

# Same question as the weak_options cell — compare the two outputs directly.
await asyncio.to_thread(run_sdk, ask(grounded_options, "My car's in the shop after an accident — does my policy cover a rental car?"))


🔧 called ToolSearch({'query': 'select:mcp__cx_tools__kb_search', 'max_results': 5})
🔧 called mcp__cx_tools__kb_search({'query': 'rental car coverage after accident'})
💬 Rental cost not cover by default. Collision [POL-4.3] and Comprehensive [POL-4.2] skip rental reimbursement.

Only cover if Rider R-12 bought — pays INR 1,500/day, cap 30 days, while car under repair for covered claim [POL-9.1].

Check policy doc: R-12 attached or nah?


## Step 5 — Persona & tone

Step 4 fixed *what* the agent is allowed to say — grounded, cited, no invented coverage. This step
is a separate concern: *how* it says it. Two answers can cite the identical clause and be equally
correct while one reads like a form letter and the other reads like someone who actually registered
that the customer's car is in the shop. CX quality isn't fully captured by factual correctness —
tone is graded too, by the customer, every time.

`persona_options` reuses `grounded_options.system_prompt` verbatim and appends a voice
specification: acknowledge the situation before reciting policy, translate clause language into
plain English, stay concise, never sound legalistic even while citing a clause id. Nothing about
retrieval or citation changes — same tool, same grounding rules, same facts.

**Why this is a separate, appended block rather than rewriting the grounding prompt:** persona and
grounding fail independently and should be debuggable independently. If an answer is wrong, that's
a retrieval or grounding problem (Steps 1–4). If an answer is correct but reads cold or robotic,
that's a persona problem (this step). Merging both concerns into one undifferentiated system prompt
makes every bug report ambiguous about which layer to fix.

**Why this block is hardcoded here and wouldn't be in production:** a real deployment pulls persona
from a brand-voice config — one per lane, versioned and owned by CX/brand, not baked into agent
code. That's Day 4 territory (policy-as-config); today's version hardcodes it so the mechanism is
visible before the abstraction is.

🔍 **Try it yourself:** run the identical rental-car question from Step 4 against `persona_options`
and compare the two outputs side by side. Same citation, same facts — read for tone, not content.


In [8]:
persona_options = ClaudeAgentOptions(
    # Built by APPENDING to grounded_options.system_prompt, not rewriting it — grounding
    # and persona are meant to fail independently and be debugged independently.
    # Caution: if grounded_options gets edited later and this cell isn't re-run after,
    # persona_options silently keeps the OLD grounding text baked into this concatenation.
    system_prompt=grounded_options.system_prompt + (
        " Voice and tone: acknowledge the customer's situation in one short sentence before "
        "stating coverage facts. Translate policy language into plain English — never make the "
        "customer parse clause-speak themselves, even while citing the clause id. Keep the whole "
        "answer under 80 words. Warm, direct, never legalistic, never apologetic filler."
    ),
    mcp_servers={"cx_tools": cx_tools_server},
    allowed_tools=["mcp__cx_tools__kb_search"],
    disallowed_tools=BUILTIN_LOCKDOWN,
)

# Same facts as grounded_options should come back — read this one for TONE, not content.
await asyncio.to_thread(run_sdk, ask(persona_options, "My car's in the shop after an accident — does my policy cover a rental car?"))


🔧 called ToolSearch({'query': 'select:mcp__cx_tools__kb_search', 'max_results': 5})
🔧 called mcp__cx_tools__kb_search({'query': 'rental car coverage after accident'})
💬 Rough one, car in shop after accident. Base policy no help there — collision [POL-4.3] and comprehensive [POL-4.2] don't cover rental cars. Only covered if got Rider R-12 [POL-9.1]: pays ₹1,500/day, max 30 days, while car repaired. Check if you have R-12 on policy.


## Step 6 — Containment & Escalation

**Containment and escalation are two names for the same design decision, seen from opposite sides:** containment is every question Steps 1–5 already resolve on their own — grounded, cited, on-brand, no human needed. Escalation is the deliberate, structured exit for everything that shouldn't be contained: out of scope, or the customer explicitly wants a person. A good agent isn't the one that contains the most — it's the one whose contain/escalate line matches reality. Over-containing means confidently answering things it shouldn't (Step 3's failure mode); over-escalating means punting questions it could have safely resolved.

This step adds `escalate_to_human`, a tool for anything outside the knowledge base's scope, and extends the system prompt so the agent hands off instead of guessing when `kb_search` doesn't return an answer.

**Why this is a structured handoff, not a form apology:** the tool takes a `reason` and a
`conversation_summary` — in production this would push to a real helpdesk/CRM queue via MCP with
full context attached. The point isn't the stub return value here; it's that escalation should
carry enough information that whoever picks it up doesn't have to re-ask the customer everything
from scratch. A generic "I'll connect you with someone" is a failed escalation even if the tool
technically fired.

**A structural note worth tracking from here on:** `escalation_options`'s system prompt is built as
`persona_options.system_prompt` plus new text — so the voice from Step 5 carries forward automatically into escalations,
actions, and instrumentation, without being re-specified at every stage. Every stage from this
point forward (Lab 2, Lab 3) extends the prompt of the stage before it the same way. That means the
notebook has to run top-to-bottom, once, without skipping around: if an earlier cell (say,
`grounded_options`) gets edited and re-run without re-running everything after it, the later agents
silently keep the *old* prompt text baked into their `..._options.system_prompt` concatenation.

🔍 **Try it yourself** with an out-of-scope question, e.g. *"Does my auto policy cover a knee
surgery I need?"* Expect `kb_search` to return nothing relevant and `escalate_to_human` to fire with
a real summary of the actual conversation — check the summary's content, not just that the tool was
called.

✅ **Checkpoint — Lab 1 complete.** This is a shippable resolution agent: grounded, on-voice
answers with correct citations, and clean containment/escalation for anything out of scope. Save
the notebook here. Banking, Retail, and Telecom variants of this knowledge base are in the
appendix — each is a drop-in replacement for `policy_chunks` only; the tool, the grounding
instruction, the persona block, and the escalation logic don't change.

**On the H1 vs. Lab split:** the lab spec's H1 groups knowledge base, action, and
escalation into one ship criterion; this notebook deliberately keeps action out of Lab 1
and behind its own approval gate in Lab 2 below — stronger isolation between guardrails,
not a missed requirement.

---

## Lab 2 — Idempotent actions with an approval gate

**Ship criterion:** an action tool that requires explicit confirmation before it fires, and that
doesn't double-execute on retry — demonstrated by the agent itself driving a preview → confirm
conversation across two turns, not by calling the tool function directly.

We extend the same agent with a "file a claim" action. (Banking's equivalent is a card-block flow —
identical pattern, different scenario.)

In [9]:
@tool(
    "escalate_to_human",
    "Hand off the conversation to a human agent when the question is outside policy scope, the "
    "customer explicitly asks for a human, or the retrieved clauses don't resolve the question.",
    {"reason": str, "conversation_summary": str},
)
async def escalate_to_human(args):
    # In production: push to your helpdesk/CRM queue via MCP, with full context.
    # The stub return value below is not the point — what matters is that `args` carries
    # enough context (reason + summary) that a human doesn't have to re-ask the customer.
    ticket = {"reason": args["reason"], "summary": args["conversation_summary"], "status": "escalated"}
    return {"content": [{"type": "text", "text": f"Escalated: {ticket}"}]}

# Re-registering cx_tools_server with the new tool added — MUST re-run this, the OLD
# server object (built back in Step 2 with only kb_search) is still bound inside
# weak_options/grounded_options/persona_options and is untouched by this reassignment.
cx_tools_server = create_sdk_mcp_server(
    name="cx_tools",
    version="1.0.0",
    tools=[kb_search, escalate_to_human]
)

escalation_options = ClaudeAgentOptions(
    # persona_options.system_prompt + one more clause — same cumulative-layering pattern
    # as persona_options itself. Worth naming: this is readable for a teaching notebook,
    # fragile in a real codebase (usually replaced by composing named prompt sections).
    system_prompt=persona_options.system_prompt + (
        " If kb_search doesn't return clauses that answer the question, call escalate_to_human "
        "with a full summary of the conversation — never guess."
    ),
    mcp_servers={"cx_tools": cx_tools_server},
    allowed_tools=["mcp__cx_tools__kb_search", "mcp__cx_tools__escalate_to_human"],
    disallowed_tools=BUILTIN_LOCKDOWN,
)

# Out-of-scope question — expect kb_search to return nothing useful, then escalate_to_human
# to fire with a real summary, not a placeholder.
await asyncio.to_thread(run_sdk, ask(escalation_options, "Does my auto policy cover a knee surgery I need?"))


🔧 called ToolSearch({'query': 'select:kb_search,escalate_to_human', 'max_results': 5})
🔧 called ToolSearch({'query': 'select:mcp__cx_tools__kb_search,mcp__cx_tools__escalate_to_human', 'max_results': 5})
🔧 called mcp__cx_tools__kb_search({'query': 'auto policy coverage medical expenses knee surgery injury'})
💬 Clauses back don't touch injury/medical stuff — only rental/collision [POL-4.2], [POL-4.3]. No answer here. Escalating.
🔧 called mcp__cx_tools__escalate_to_human({'reason': 'KB has no clauses on medical/injury/personal-accident coverage under auto policy; only rental reimbursement and collision damage clauses returned.', 'conversation_summary': 'Customer asked if their auto policy covers knee surgery they need. kb_search for "auto policy coverage medical expenses knee surgery injury" returned only POL-4.2 (rental reimbursement), POL-4.3 (collision, vehicle-to-vehicle), POL-9.1 (Rider R-12 rental cost cap) — none address personal injury/medical/surgery coverage. Cannot answer from

### The `file_claim` tool

Two dictionaries stand in for real storage: `pending_claims` (idempotency key → preview args) and
`filed_claims` (idempotency key → claim id).

**Reading the logic, in precedence order:**
1. If this idempotency key has already filed something, return the existing claim — never refile.
2. If `confirm` is `False`, store the args as a pending preview and return preview text only —
   nothing gets filed yet.
3. Only if `confirm` is `True` *and* a matching pending preview exists does the claim actually get
   filed and recorded against the key.

**Why the approval gate is structural rather than conversational:** there is no code path in this
function that files a claim without a prior preview under the same key. That's a stronger guarantee
than a well-worded prompt asking the model to "confirm before filing" — a prompt can be ignored by
a model under the right (or wrong) circumstances; a missing code path can't be bypassed the same
way.

**Why idempotency matters here specifically:** if the network hiccups and the same confirm call
fires twice with the same key, the customer ends up with one claim, not two. That guarantee comes
entirely from the `key in filed_claims` check at the top — remove it and a retried request becomes
a duplicate claim.


In [10]:
pending_claims = {}   # idempotency_key -> preview args (swap for real storage in production)
filed_claims = {}      # idempotency_key -> claim id

@tool(
    "file_claim",
    "File an insurance claim. Two-step tool: call with confirm=False first to preview, then "
    "confirm=True only after the customer has explicitly agreed to the details shown.",
    {
        "policy_id": str,
        "incident_description": str,
        "estimated_amount": float,
        "confirm": bool,
        "idempotency_key": str,
    },
)
async def file_claim(args):
    key = args["idempotency_key"]

    # Precedence 1: already filed under this key -> return the EXISTING claim, never refile.
    # This is what makes retries/duplicate calls safe (idempotency).
    if key in filed_claims:
        return {"content": [{
            "type": "text",
            "text": f"Already filed: {filed_claims[key]}"
            }
            ]}

    # Precedence 2: confirm=False -> store the preview, file NOTHING yet.
    if not args["confirm"]:
        preview = (f"Preview — policy {args['policy_id']}, est. amount INR "
                   f"{args['estimated_amount']}. Ask the customer to confirm before filing.")
        pending_claims[key] = args
        return {"content": [{"type": "text", "text": preview}]}

    # Precedence 3: confirm=True but no matching preview under this key -> refuse.
    # There is no code path that files a claim without a prior preview under the SAME key —
    # that's a structural guarantee, not a prompt asking the model nicely to confirm first.
    if key not in pending_claims:
        return {"content": [{"type": "text",
                 "text": "No pending preview found for this key — preview first."}]}

    # Only reachable with a valid, previewed key and confirm=True — actually files the claim.
    claim_id = f"CLM-{len(filed_claims) + 1000}"
    filed_claims[key] = claim_id
    return {"content": [{"type": "text",
             "text": f"Claim {claim_id} filed for policy {args['policy_id']}."}]}

print("file_claim tool defined.")


file_claim tool defined.


## Lab 2 — Multi-turn confirm flow

`file_claim` is a two-step tool: `confirm=False` previews, `confirm=True` files — and only after
an explicit customer confirmation. That means this turn's response has to be informed by what the
model said (and generated — the `idempotency_key`) on the *previous* turn.

`ask()` can't do that: it opens a fresh `ClaudeSDKClient` session per call, so turn 2 would start
with no memory of turn 1's preview. The next cell keeps a single `ClaudeSDKClient` open across both
`client.query()` calls instead, which is what lets the model reuse the same `idempotency_key` on
turn 2 rather than generating a new one.


In [11]:
cx_tools_server = create_sdk_mcp_server(
    name="cx_tools", version="1.0.0", tools=[kb_search, escalate_to_human, file_claim],
)

action_options = ClaudeAgentOptions(
    system_prompt=escalation_options.system_prompt + (
        " When a customer asks you to file a claim: call file_claim with confirm=False first to "
        "show a preview, generating a short idempotency_key for this claim attempt. After the "
        "customer explicitly confirms, call file_claim again with confirm=True, reusing the EXACT "
        "SAME idempotency_key from the preview — never generate a new one for the same claim."
    ),
    mcp_servers={"cx_tools": cx_tools_server},
    allowed_tools=[
        "mcp__cx_tools__kb_search",
        "mcp__cx_tools__escalate_to_human",
        "mcp__cx_tools__file_claim",
    ],
    disallowed_tools=BUILTIN_LOCKDOWN,
)

# Using ClaudeSDKClient directly here instead of ask() — deliberately. ask() opens and
# closes a NEW session every call, so turn 2 would have zero memory of turn 1's preview
# or its idempotency_key. Keeping ONE client open across both client.query() calls is
# what lets the model recall and reuse the key it generated on turn 1, on turn 2.
async def _lab2_file_claim_flow():
    async with ClaudeSDKClient(options=action_options) as client:
        print("--- turn 1: customer asks to file a claim ---")
        await client.query(
            "I was in a fender bender yesterday, my policy id is POL-100 and the repair "
            "estimate is INR 25000. Can you file a claim for me?"
        )
        async for message in client.receive_response():
            content = getattr(message, "content", None)
            if content:
                for block in content:
                    if hasattr(block, "text"):
                        print(f"\U0001F4AC {block.text}")
                    elif hasattr(block, "name"):
                        print(f"\U0001F527 called {block.name}({block.input})")

        print("\n--- turn 2: customer confirms ---")
        await client.query("Yes, please go ahead and file it.")   # same client -> same session
        async for message in client.receive_response():
            content = getattr(message, "content", None)
            if content:
                for block in content:
                    if hasattr(block, "text"):
                        print(f"\U0001F4AC {block.text}")
                    elif hasattr(block, "name"):
                        print(f"\U0001F527 called {block.name}({block.input})")

# Routed through run_sdk() via a worker thread -- same Windows subprocess-spawn fix as ask() above;
# this cell used a bare top-level `async with` before and crashed on Windows the same way ask() did.
await asyncio.to_thread(run_sdk, _lab2_file_claim_flow())

print("\nfiled_claims so far:", filed_claims)


--- turn 1: customer asks to file a claim ---
🔧 called ToolSearch({'query': 'select:mcp__cx_tools__file_claim', 'max_results': 3})
🔧 called mcp__cx_tools__file_claim({'policy_id': 'POL-100', 'incident_description': 'Fender bender, yesterday', 'estimated_amount': 25000, 'confirm': False, 'idempotency_key': 'POL-100-fenderbender-20260724'})
💬 Sorry bout wreck. Here preview: policy POL-100, damage est INR 25000, fender bender yesterday. Confirm file now?

--- turn 2: customer confirms ---
🔧 called mcp__cx_tools__file_claim({'policy_id': 'POL-100', 'incident_description': 'Fender bender, yesterday', 'estimated_amount': 25000, 'confirm': True, 'idempotency_key': 'POL-100-fenderbender-20260724'})
💬 Claim filed — CLM-1000, policy POL-100. Adjuster reach out soon.

filed_claims so far: {'POL-100-fenderbender-20260724': 'CLM-1000'}


✅ **Checkpoint check** — before moving on, confirm all three of these hold in the output above:
turn 1 produced a preview without filing anything, turn 2 actually filed it, and `filed_claims`
contains exactly one entry.

### A faster, offline check on the same logic

The cell below re-verifies the idempotency logic directly — no API call, no model involved. It's
useful any time the tool logic itself changes and a quick recheck is needed without spending API
calls. One detail: it calls `file_claim.handler(...)`, not `file_claim(...)` — the `@tool` decorator
wraps the function in an `SdkMcpTool` object, so the raw function isn't directly callable anymore.
The third call retries the same key with `confirm=True` — it should return the *original* claim id,
not create a new one.

✅ **Checkpoint — Lab 2 complete.** Save the notebook here.

**Banking variant:** the same pattern as `block_card(card_id, confirm, idempotency_key)` — preview
shows the last 4 digits and a reason, confirm actually blocks the card, and a retry with the same
key returns "already blocked."

---

## Lab 3 — Resolution and escalation instrumentation

**Ship criterion:** every test conversation gets logged as `resolved`, `escalated`, or `failed` —
by the agent itself, autonomously, not by calling the logging function on its behalf. This log is
the raw data Day 5's evaluation framework consumes later in the course, so how carefully this gets
logged now determines how meaningful that later eval is.


In [12]:
# Direct logic check — no API call needed. Note the .handler(...) call, not file_claim(...) —
# the @tool decorator wraps the function in an SdkMcpTool object, so the raw function isn't
# directly callable anymore; .handler(args) is how you invoke the underlying async function.
#
# What this DOESN'T test: whether the MODEL itself would generate and reuse the same
# idempotency_key across two real conversational turns — that's a property of the agent
# driving the tool, not of the tool's Python logic, and no offline check can catch it.
# The two-turn cell above is what actually exercises that failure mode.
async def _test_file_claim_logic():
    key = "demo-key-001"
    preview = await file_claim.handler({"policy_id": "POL-200", "incident_description": "test",
                                         "estimated_amount": 10000.0, "confirm": False,
                                         "idempotency_key": key})
    print("1) preview:", preview["content"][0]["text"])

    confirmed = await file_claim.handler({"policy_id": "POL-200", "incident_description": "test",
                                           "estimated_amount": 10000.0, "confirm": True,
                                           "idempotency_key": key})
    print("2) confirmed:", confirmed["content"][0]["text"])

    # Same key, confirm=True again — precedence 1 in file_claim should return the EXISTING
    # claim id below, not mint a new CLM-XXXX.
    retried = await file_claim.handler({"policy_id": "POL-200", "incident_description": "test",
                                         "estimated_amount": 10000.0, "confirm": True,
                                         "idempotency_key": key})
    print("3) retried (should NOT be a new claim):", retried["content"][0]["text"])

await _test_file_claim_logic()


1) preview: Preview — policy POL-200, est. amount INR 10000.0. Ask the customer to confirm before filing.
2) confirmed: Claim CLM-1001 filed for policy POL-200.
3) retried (should NOT be a new claim): Already filed: CLM-1001


### The `log_outcome` tool

One tool, one responsibility: log the final outcome of a conversation, exactly once, at the end.
The `assert` on `outcome` keeps the log clean — only `resolved`, `escalated`, or `failed` are valid
values, nothing else gets written.


In [13]:
conversation_log = []   # append-only, in-memory — swap for real storage/telemetry in production

@tool(
    "log_outcome",
    "Log the final outcome of this conversation. Call this exactly once, at the end, before "
    "ending the turn.",
    {"outcome": str, "lane": str, "tools_called": str, "notes": str},
)
async def log_outcome(args):
    # Hard constraint on the value set — keeps the log clean at write time rather than
    # needing to clean bad values out later. This is the ONLY validation this tool does;
    # it does NOT check whether "resolved" is actually true (see markdown below).
    assert args["outcome"] in ("resolved", "escalated", "failed"), \
        "outcome must be resolved | escalated | failed"
    entry = {**args, "ts": time.time()}
    conversation_log.append(entry)
    return {"content": [{"type": "text", "text": f"Logged: {json.dumps(entry)}"}]}

print("log_outcome tool defined.")


log_outcome tool defined.


### Wiring it in — two real conversations, logged live

**Why the log fills up in real time:** `create_sdk_mcp_server` runs in-process, so when the agent
calls `log_outcome`, it's calling the actual Python function directly at that moment —
`conversation_log` grows as the agent decides to log, not after the fact by some separate process
reading transcripts later.

**The distinction this lab is really testing:** logging a conversation as `resolved` because the
agent *produced an answer* is the same failure mode as a deflection-optimized chatbot that closes
tickets to hit a metric. `resolved` should mean the customer's actual problem got fixed — not that
the conversation ended.

Run conversation A (the rental car question — should resolve) and conversation B (the knee surgery
question — should escalate), and check that `conversation_log` ends up with both entries.


In [14]:
cx_tools_server = create_sdk_mcp_server(
    name="cx_tools",
    version="1.0.0",
    tools=[kb_search, escalate_to_human, file_claim, log_outcome],
)

full_options = ClaudeAgentOptions(
    system_prompt=action_options.system_prompt + (
        " At the end of every conversation, call log_outcome exactly once before ending your "
        "turn. 'resolved' means the customer's actual problem was fixed — not just that the "
        "conversation ended. Use 'escalated' when you handed off to a human, 'failed' if you "
        "could not help and did not escalate."
    ),
    mcp_servers={"cx_tools": cx_tools_server},
    allowed_tools=[
        "mcp__cx_tools__kb_search",
        "mcp__cx_tools__escalate_to_human",
        "mcp__cx_tools__file_claim",
        "mcp__cx_tools__log_outcome",
    ],
    disallowed_tools=BUILTIN_LOCKDOWN,
)

# Each ask() call below is its OWN session (see the ask() helper in Step 2) — conversation B
# starts with zero memory of conversation A. That's fine for this lab: the point is that
# EACH conversation ends with exactly one log_outcome call, not that they share context.
print("--- conversation A: should resolve ---")
await asyncio.to_thread(run_sdk, ask(full_options, "My car's in the shop after an accident — does my policy cover a rental car?"))

print("\n--- conversation B: should escalate ---")
await asyncio.to_thread(run_sdk, ask(full_options, "Does my auto policy cover a knee surgery I need?"))

print("\nconversation_log:")
for entry in conversation_log:
    print(entry)


--- conversation A: should resolve ---
🔧 called ToolSearch({'query': 'select:mcp__cx_tools__kb_search', 'max_results': 1})
🔧 called mcp__cx_tools__kb_search({'query': 'rental car coverage after accident'})
💬 Rough day, car in shop. Straight answer: base policy no cover rental car [POL-4.2][POL-4.3]. Only covered if got Rider R-12 — pays ₹1,500/day, max 30 days, while car repaired for covered claim [POL-9.1]. Check policy doc if R-12 attached.
🔧 called ToolSearch({'query': 'select:mcp__cx_tools__log_outcome', 'max_results': 1})
🔧 called mcp__cx_tools__log_outcome({'outcome': 'resolved', 'lane': 'coverage_question', 'tools_called': 'kb_search', 'notes': 'Customer asked if policy covers rental car after accident. kb_search returned POL-4.2, POL-4.3, POL-9.1. Explained base policy excludes rental reimbursement unless Rider R-12 purchased (covers up to INR 1,500/day, 30 day cap, while insured vehicle under repair for covered claim). Advised customer to check if R-12 attached. No further act

✅ **Checkpoint check** — confirm `conversation_log` has two entries above: one `resolved`, one
`escalated`, each with a genuinely useful `notes` field (not a placeholder).

🔍 **Try it yourself:** produce a third conversation that should log as `failed`. This is less
obvious than it sounds, because the agent already escalates anything outside the knowledge base's
scope — a plain out-of-scope question will just log `escalated` again, not `failed`. A `failed`
case needs to be something the agent has *no tool for and no clean way to hand off usefully* — for
example: *"Can you just go ahead and cancel my policy right now?"* There's no `cancel_policy` tool
and no clearly-scoped escalation path for a direct account action like this. Whatever the agent
actually does with it is informative either way: if it correctly logs `failed`, good; if it logs
`escalated` by default for everything it can't do, that's a sign the system prompt's distinction
between `escalated` ("needs a human, here's useful context") and `failed` ("needs a human and I
can't even frame it usefully") is too soft, and worth tightening.

### A faster, offline check on the logging contract

Same idea as Lab 2's offline check — no API call, just verifying `log_outcome`'s behavior directly.
Useful for a quick recheck any time the logging contract itself changes.

✅ **Checkpoint — Lab 3 complete.** Save the notebook here.


In [15]:
# Direct logic check — no API call needed. Same purpose as Lab 2's offline check: fast
# recheck of the logging CONTRACT (does log_outcome accept/reject the right values, does
# it append correctly) without spending an API call or waiting on the model.
async def _test_log_logic():
    await log_outcome.handler({"outcome": "resolved", "lane": "insurance",
                                "tools_called": "kb_search", "notes": "rental car coverage question"})
    await log_outcome.handler({"outcome": "escalated", "lane": "insurance",
                                "tools_called": "kb_search,escalate_to_human",
                                "notes": "out of scope: health"})
    for entry in conversation_log[-2:]:
        print(entry)

await _test_log_logic()


{'outcome': 'resolved', 'lane': 'insurance', 'tools_called': 'kb_search', 'notes': 'rental car coverage question', 'ts': 1784877414.797445}
{'outcome': 'escalated', 'lane': 'insurance', 'tools_called': 'kb_search,escalate_to_human', 'notes': 'out of scope: health', 'ts': 1784877414.797471}


## Closing out

Before wrapping up, look back over your own test conversations from today and, for each one, decide
plainly: did it resolve the customer's actual problem, or did it just produce a response that ended
the conversation? That distinction — resolution versus deflection — is the idea the rest of this
course builds on, so it's worth being honest about it now rather than assuming every completed
conversation counts as a win.

---

## Ship rubric

| Checkpoint | Pass criteria |
|---|---|
| Lab 1 | Agent answers a KB question with a correct citation to the specific clause |
| Lab 1 | An out-of-scope question escalates cleanly instead of getting a guessed answer |
| Lab 1 | Grounded answers read in the defined voice (warm, plain-language, concise) — not just factually correct |
| Lab 2 | The agent itself drives confirm before firing; the same key called twice doesn't duplicate |
| Lab 3 | The agent itself logs every test conversation as resolved / escalated / failed — nothing silent |
| Closing | You can correctly state, for your own test cases, whether each one resolved or merely deflected |

## Known-breakage cheat sheet

| Symptom | Likely cause | Fix |
|---|---|---|
| `CLINotFoundError` on first run | Claude Code CLI not installed | `npm install -g @anthropic-ai/claude-code` |
| `SyntaxError: 'await' outside function` | Kernel doesn't support top-level await, or a cell was copied into a plain `.py` file | Confirm kernel version during setup; run cells only inside the notebook |
| Tool call never fires / hangs | `allowed_tools` missing the exact `mcp__<server>__<tool>` name | Check the server name in `mcp_servers` matches the prefix used in `allowed_tools` |
| `TypeError: 'SdkMcpTool' object is not callable` | Called the decorated tool directly instead of `.handler(...)` | Only use `.handler(...)` for offline checks — never call the tool object itself |
| Agent answers without calling `kb_search` | Tool description doesn't mandate it, or the system prompt still allows general knowledge | Tighten the "MUST call" language in both places |
| Citation present but wrong clause | Keyword scorer pulled a loosely-related chunk | Widen `top_k` or use a more specific query; in production this is an embedding-quality problem |
| Confirm call rejected as "no pending preview" | Model generated a *different* `idempotency_key` on the confirm turn | Reinforce the reuse instruction in the system prompt — a real teaching moment, not just a bug to hide |
| `log_outcome` never called | System prompt doesn't make it a hard requirement | Add "call exactly once before ending the turn" |
| Later agent behaves like an earlier, less-capable version | An earlier cell (e.g. `grounded_options`) was edited and re-run without re-running every cell after it, so a later `..._options.system_prompt` concatenation is built on stale text | Always re-run top-to-bottom after editing any earlier cell — don't skip around |

---

## Appendix — lane variants for the Lab 1 knowledge base

**Banking:** chunks on dispute windows, overdraft fee waivers, card replacement timelines.
Scripted-failure question: the agent gives a generic "most banks refund unauthorized transactions
within 10 days" instead of citing this policy's actual 45-day window.

**Retail:** chunks on return windows, restocking fees, final-sale exclusions. Scripted-failure
question: the agent assumes a standard 30-day return window when this retailer's policy is 14 days
for electronics.

**Telecom:** chunks on early-termination fees, data rollover rules, SIM-swap security holds.
Scripted-failure question: the agent states data rollover is automatic when this carrier actually
requires an opt-in add-on.

Each swap is a drop-in replacement for `policy_chunks` only — the tool, the grounding instruction,
the persona block, and the escalation logic are unchanged.


---

## Appendix — Retail: resolution + escalation instrumentation ("measured agent")

The KB-appendix note above only swaps `policy_chunks` for a Retail catalog — it doesn't show
the domain running end-to-end. This section makes that concrete: a Retail-lane agent, wired with
its own knowledge base and its own `kb_search` tool, that reuses the exact same
`escalate_to_human` and `log_outcome` tools defined in Step 6 and Lab 3 (both are already
domain-generic — `log_outcome`'s schema even carries a `lane` field for exactly this).
Nothing here is chained off `persona_options` or `action_options` on purpose: this subsection can
run on its own, right after Step 1, without depending on Labs 1–3 having been run first.

**Scripted-failure clause this KB is built around:** this retailer's return window for
electronics is 14 days, not the industry-standard 30 days a model might assume from general
knowledge — the same trap as Step 3's rental-car question, just in a different domain.


In [ ]:
retail_chunks = [
    {"id": "RET-1.1", "text": "Standard merchandise may be returned within 30 days of purchase "
     "for a full refund, provided the item is unused and in its original packaging."},
    {"id": "RET-1.2", "text": "Electronics (phones, laptops, headphones and cameras) have a "
     "return window of 14 days from purchase, not the standard 30 days, due to rapid "
     "depreciation."},
    {"id": "RET-2.4", "text": "Opened software, digital downloads, and gift cards are final "
     "sale and not eligible for return or refund under any circumstances."},
    {"id": "RET-3.1", "text": "A 15% restocking fee applies to any returned item missing its "
     "original packaging or accessories, deducted from the refund amount."},
]

def retail_search(query: str, top_k: int = 3):
    # Reuses score() from Step 1 -- (query, text) -> float -- only the corpus changes.
    ranked = sorted(retail_chunks, key=lambda c: score(query, c["text"]), reverse=True)
    return ranked[:top_k]

# Sanity check, same purpose as Step 1's: the electronics question must surface RET-1.2, not
# the generic RET-1.1 30-day clause -- that's the scripted-failure this KB is built around.
for r in retail_search("can I return my headphones after 20 days"):
    print(r["id"], "-", r["text"])


In [ ]:
@tool(
    "kb_search_retail",
    "Search the retail return-policy knowledge base for clauses relevant to a customer's "
    "question. Always call this before answering any question about returns, refunds, or fees.",
    {"query": str},
)
async def kb_search_retail(args):
    results = retail_search(args["query"])
    formatted = "\n".join(f"[{r['id']}] {r['text']}" for r in results)
    return {"content": [{"type": "text", "text": formatted}]}

# Reuses the escalate_to_human and log_outcome SdkMcpTool objects defined earlier verbatim --
# both are domain-generic already, no Retail-specific rewrite needed.
retail_tools_server = create_sdk_mcp_server(
    name="retail_tools",
    version="1.0.0",
    tools=[kb_search_retail, escalate_to_human, log_outcome],
)

retail_options = ClaudeAgentOptions(
    # Standalone system prompt -- deliberately NOT persona_options.system_prompt + "...",
    # so this subsection has no dependency on Labs 1-3 having run first.
    system_prompt=(
        "You are a retail customer support agent. For ANY question about returns, refunds, or "
        "fees, you MUST call kb_search_retail first and answer only from the returned clauses "
        "-- never from general knowledge about typical return policies. Always cite the clause "
        "id(s) you used. If the retrieved clauses don't answer the question, or the customer "
        "explicitly asks for a human/supervisor, call escalate_to_human with a real summary of "
        "the conversation -- never guess. At the end of every conversation, call log_outcome "
        "exactly once before ending your turn, with lane='retail'. 'resolved' means the "
        "customer's actual question was answered correctly with a citation; use 'escalated' "
        "when you handed off to a human, 'failed' if you could not help and did not escalate. "
        "Be concise and warm."
    ),
    mcp_servers={"retail_tools": retail_tools_server},
    allowed_tools=[
        "mcp__retail_tools__kb_search_retail",
        "mcp__retail_tools__escalate_to_human",
        "mcp__retail_tools__log_outcome",
    ],
    disallowed_tools=BUILTIN_LOCKDOWN,
)

print("retail_tools_server registered.")


In [ ]:
print("--- retail conversation A: should resolve ---")
await asyncio.to_thread(run_sdk, ask(retail_options, "Can I return my headphones after 20 days?"))

print("\n--- retail conversation B: should escalate ---")
await asyncio.to_thread(run_sdk, ask(
    retail_options,
    "This is unacceptable, let me talk to a supervisor about my order.",
))

print("\nconversation_log (retail lane):")
for entry in conversation_log:
    if entry.get("lane") == "retail":
        print(entry)


✅ **Checkpoint — Retail ships a measured agent.** Confirm exactly one `resolved` and one
`escalated` entry above with `lane: retail`, each with a real, non-placeholder `notes` field --
the same bar Lab 3's own checkpoint uses. Conversation A should cite `RET-1.2` (the 14-day
electronics window), not the generic 30-day `RET-1.1` clause. This is the literal H3 ship
criterion from the lab spec: resolution/escalation instrumentation on the Retail lane, not just
Insurance.


---

## Appendix — Banking: idempotent action + approval gate ("safe-action agent")

`block_card` (in the HandsOnExercise track) is confirm-gated by system-prompt wording alone --
strip that instruction out and the model will sometimes call the tool on the very first message,
no confirmation asked. This section closes that gap the same way Lab 2's `file_claim` already
does for Insurance: the approval gate becomes a property of the code, not the prompt -- there is
no path through `block_card_banking` that blocks a card without a prior preview under the same
`idempotency_key`, no matter what the model does or doesn't say. Same 3-precedence pattern as
`file_claim`, same reason it matters: a retried confirm call with the same key must not
double-block or mint a second confirmation number. Standalone system prompt again, on purpose --
no dependency on Labs 1–3.


In [ ]:
pending_blocks = {}   # idempotency_key -> preview args
blocked_cards = {}    # idempotency_key -> confirmation number

@tool(
    "block_card_banking",
    "Permanently block a debit/credit card. Two-step tool: call with confirm=False first to "
    "preview, then confirm=True only after the customer has explicitly agreed to proceed.",
    {
        "card_last4": str,
        "reason": str,
        "confirm": bool,
        "idempotency_key": str,
    },
)
async def block_card_banking(args):
    key = args["idempotency_key"]

    # Precedence 1: already blocked under this key -> return the EXISTING confirmation, never
    # re-block. This is what makes a retried confirm call safe.
    if key in blocked_cards:
        return {"content": [{"type": "text", "text": f"Already blocked: {blocked_cards[key]}"}]}

    # Precedence 2: confirm=False -> store the preview, block NOTHING yet.
    if not args["confirm"]:
        preview = (f"Preview -- block card ending {args['card_last4']}, reason: "
                   f"{args['reason']}. Ask the customer to confirm before blocking.")
        pending_blocks[key] = args
        return {"content": [{"type": "text", "text": preview}]}

    # Precedence 3: confirm=True but no matching preview under this key -> refuse. There is no
    # code path that blocks a card without a prior preview under the SAME key.
    if key not in pending_blocks:
        return {"content": [{"type": "text",
                 "text": "No pending preview found for this key -- preview first."}]}

    # Only reachable with a previewed key and confirm=True -- actually blocks the card.
    confirmation = f"BLK-{len(blocked_cards) + 88200}"
    blocked_cards[key] = confirmation
    return {"content": [{"type": "text",
             "text": f"Card ending {args['card_last4']} blocked. Confirmation: {confirmation}."}]}

banking_tools_server = create_sdk_mcp_server(
    name="banking_tools", version="1.0.0", tools=[block_card_banking, log_outcome],
)

banking_action_options = ClaudeAgentOptions(
    system_prompt=(
        "You are a banking support agent. You have access to block_card_banking, which "
        "PERMANENTLY and IRREVERSIBLY blocks a card. Never call it on the first mention of a "
        "lost/stolen card. First call it with confirm=False to preview, generating a short "
        "idempotency_key for this attempt. Only after the customer gives clear, explicit "
        "confirmation in a LATER message, call it again with confirm=True, reusing the EXACT "
        "SAME idempotency_key from the preview -- never generate a new one for the same "
        "request. After it executes, tell the customer it's done, give them the confirmation "
        "number, and mention a replacement card typically arrives in 5-7 business days. At the "
        "end of the conversation, call log_outcome exactly once with lane='banking'. Be concise "
        "and reassuring -- the customer is likely stressed about fraud."
    ),
    mcp_servers={"banking_tools": banking_tools_server},
    allowed_tools=["mcp__banking_tools__block_card_banking", "mcp__banking_tools__log_outcome"],
    disallowed_tools=BUILTIN_LOCKDOWN,
)

print("banking_tools_server registered.")


In [ ]:
async def _banking_block_card_flow():
    async with ClaudeSDKClient(options=banking_action_options) as client:
        print("--- turn 1: customer reports a stolen card ---")
        await client.query("Hi, my card was stolen, please block it right now.")
        async for message in client.receive_response():
            content = getattr(message, "content", None)
            if content:
                for block in content:
                    if hasattr(block, "text"):
                        print(f"💬 {block.text}")
                    elif hasattr(block, "name"):
                        print(f"🔧 called {block.name}({block.input})")

        print("\n--- turn 2: customer confirms ---")
        await client.query("Yes, it's the one ending 4471, please block it.")
        async for message in client.receive_response():
            content = getattr(message, "content", None)
            if content:
                for block in content:
                    if hasattr(block, "text"):
                        print(f"💬 {block.text}")
                    elif hasattr(block, "name"):
                        print(f"🔧 called {block.name}({block.input})")

await asyncio.to_thread(run_sdk, _banking_block_card_flow())
print("\nblocked_cards so far:", blocked_cards)


In [ ]:
# Direct logic check -- no API call needed, same purpose as Lab 2's offline check on file_claim.
async def _test_block_card_logic():
    key = "demo-block-001"
    preview = await block_card_banking.handler({"card_last4": "4471", "reason": "stolen",
                                                  "confirm": False, "idempotency_key": key})
    print("1) preview:", preview["content"][0]["text"])

    confirmed = await block_card_banking.handler({"card_last4": "4471", "reason": "stolen",
                                                    "confirm": True, "idempotency_key": key})
    print("2) confirmed:", confirmed["content"][0]["text"])

    # Same key, confirm=True again -- precedence 1 should return the EXISTING confirmation
    # number below, not mint a new BLK-xxxxx.
    retried = await block_card_banking.handler({"card_last4": "4471", "reason": "stolen",
                                                  "confirm": True, "idempotency_key": key})
    print("3) retried (should NOT be a new confirmation number):", retried["content"][0]["text"])

await _test_block_card_logic()


✅ **Checkpoint — Banking ships a safe-action agent.** Confirm turn 1 above did not block
anything (preview only), turn 2 blocked the card and returned a confirmation number, and the
offline check's call 3 returned the identical confirmation number as call 2 -- not a new one.
This is the literal H2 ship criterion from the lab spec: idempotent actions + an approval gate,
enforced structurally, on the Banking lane.


---

## Appendix — the same resolution-agent pattern in LangChain

Everything above was built on the Claude Agent SDK, on purpose — one continuous build thread for
the week. This section exists for **framework literacy**, not because the lab needs it: the same
grounding pattern from Steps 1–4 (retrieval tool + a system prompt that mandates using it),
rebuilt with [LangChain](https://docs.langchain.com/oss/python/langchain/agents) instead. It's
optional — skip it if you're short on time, nothing later in the week depends on it.

**Setup, and one real gotcha:** `pip install langchain langchain-anthropic`. Unlike the Claude
Agent SDK — which authenticates through the Claude Code CLI (`claude login` or
`ANTHROPIC_API_KEY`) — `ChatAnthropic` here talks to the Anthropic API **directly** and only
accepts a real `ANTHROPIC_API_KEY` environment variable. If you've only ever run `claude login`
and never set the raw env var, the cell below will fail with an auth error even though everything
above it in this notebook works fine. That's not a bug in either tool — it's two different
authentication paths that happen to look similar.

**What's identical to Step 1–4, reused verbatim:** `policy_chunks` and `search()` — the exact same
knowledge base and retrieval function defined back in Step 1. This is deliberate: same facts, same
retrieval logic, only the orchestration framework changes. If the LangChain agent below answers
the rental-car question correctly, that's proof the *grounding pattern* generalizes across
frameworks — it was never specific to the Claude Agent SDK.

▶ **Run this** (needs a real `ANTHROPIC_API_KEY`) — expect the same correctly-cited answer as
Step 4's `grounded_options`, produced by a completely different agent loop.

In [16]:
from langchain.tools import tool as lc_tool
from langchain.agents import create_agent
from langchain_anthropic import ChatAnthropic

# @tool here (langchain_core, via langchain.tools) works differently from claude_agent_sdk's
# @tool: no separate name/description/schema args — the function's NAME becomes the tool
# name, its TYPE HINTS become the input schema, and its DOCSTRING becomes the description.
# Same idea as claude_agent_sdk's tool description ("Always call this before...") — LangChain
# just reads it from the docstring instead of a separate string argument.
@lc_tool
def lc_kb_search(query: str) -> str:
    """Search the insurance policy knowledge base for clauses relevant to a customer's
    question. Always call this before answering any question about what is or isn't covered."""
    results = search(query)   # reuses Step 1's search() and policy_chunks — same KB, same logic
    return "\n".join(f"[{r['id']}] {r['text']}" for r in results)

# create_agent is LangChain's current (v1) entrypoint for building a tool-calling agent — it
# returns a compiled LangGraph graph under the hood (LangChain's agent loop IS a LangGraph
# graph; you're seeing the same underlying engine Day 2's LangGraph appendix uses directly).
lc_agent = create_agent(
    model=ChatAnthropic(model="claude-sonnet-4-6"),
    tools=[lc_kb_search],
    # Identical grounding discipline to Step 4's grounded_options — same "MUST call" mandate,
    # same citation requirement. The mechanism that enforces it (a tool-calling loop) is
    # LangGraph's, not the Claude Agent SDK's, but the PROMPT ENGINEERING doesn't change at all.
    system_prompt=(
        "You are an insurance customer service assistant. For ANY question about coverage, "
        "exclusions, or policy terms, you MUST call lc_kb_search first and answer only from "
        "the returned clauses. Always cite the clause id(s) you used, in the form [POL-x.x]. "
        "If the retrieved clauses don't fully answer the question, say so — never fill gaps "
        "from general knowledge."
    ),
)

# invoke() takes/returns a dict with a "messages" list — a different shape from the SDK's
# streamed message objects, but the same underlying turn: user message in, tool call,
# tool result, final assistant message out. result["messages"][-1] is that final message.
result = lc_agent.invoke({
    "messages": [{"role": "user",
                  "content": "My car's in the shop after an accident — does my policy cover a rental car?"}]
})
print(result["messages"][-1].content)


c:\Users\AAGATI\Desktop\Aagati_Work_Directory\AgenticCX_AppliedLabs_TrainingLabs\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Great question! Here's what your policy says about rental car coverage:

**Rental car coverage is not included by default.** Here are the key details:

1. **Standard Collision & Comprehensive Coverage** — Neither your collision coverage nor your comprehensive coverage automatically includes rental car reimbursement. [POL-4.3, POL-4.2]

2. **Rider R-12 (Rental Reimbursement)** — Rental coverage is only available if you've purchased this optional add-on separately. If you have Rider R-12, it provides: [POL-9.1]
   - Up to **₹1,500 per day** for a rental vehicle
   - Coverage for a **maximum of 30 days**
   - Only while your insured vehicle is under repair due to a **covered claim**

**Next Steps:**
- ✅ **Check your policy documents** to see if you purchased Rider R-12.
- 📞 **Contact us** and we can verify whether Rider R-12 is active on your policy.

Would you like help with anything else regarding your claim or coverage?


**What's identical, what's different:** identical — the knowledge base, the retrieval function,
the grounding mandate as a system-prompt instruction, the citation requirement. Different — the
tool-definition syntax (`@lc_tool` reads a docstring + type hints instead of taking explicit
name/description/schema arguments), the return shape (`result["messages"]`, a list of typed
message objects, instead of the SDK's streamed message events), and what's actually executing
underneath: `create_agent` compiles to a LangGraph state graph — Day 2's LangGraph appendix builds
that same kind of graph directly, by hand, instead of through this shortcut.

**The takeaway worth stating plainly:** grounding is a prompt-engineering discipline, not a
framework feature. Any tool-calling loop — the Claude Agent SDK's, LangChain's, a hand-rolled one
— fails the same way (confidently answering from parametric knowledge) without an explicit mandate
to retrieve first, and succeeds the same way once that mandate is in the system prompt. The
mechanism enforcing the loop changed; the actual fix from Step 4 did not.

### Offline wiring check — no API key needed

The cell below doesn't test whether the *model* grounds correctly — that needs a real key and
Step 4 already covers it. It tests something narrower and more mechanical: that `create_agent`'s
tool-calling loop actually wires together correctly — that a scripted "tool call" message really
does trigger `lc_kb_search`, and the tool's real output really does make it into the next model
turn. Useful for a fast recheck if this cell ever stops working and you need to know whether the
bug is in the graph wiring or in the model's behavior.


In [17]:
# Direct wiring check — no API call, no ANTHROPIC_API_KEY needed.
from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel
from langchain_core.messages import AIMessage, ToolCall

class _ScriptedToolCallingModel(FakeMessagesListChatModel):
    """FakeMessagesListChatModel doesn't implement bind_tools (BaseChatModel raises
    NotImplementedError by default) — create_agent calls bind_tools internally to wire the
    tool schema in. The scripted responses below already encode the tool_calls directly,
    so binding can safely be a no-op here."""
    def bind_tools(self, tools, **kwargs):
        return self

_fake_model = _ScriptedToolCallingModel(responses=[
    # Turn 1: the "model" decides to call lc_kb_search — scripted, not a real decision.
    AIMessage(content="", tool_calls=[
        ToolCall(name="lc_kb_search", args={"query": "rental car"}, id="call_1")
    ]),
    # Turn 2: the "model" answers using whatever lc_kb_search actually returned.
    AIMessage(content="Rental cars are addressed in the retrieved clauses above."),
])

_wiring_test_agent = create_agent(model=_fake_model, tools=[lc_kb_search], system_prompt="test")
_wiring_result = _wiring_test_agent.invoke({
    "messages": [{"role": "user", "content": "does my policy cover a rental car?"}]
})

# If the wiring is correct: 4 messages (human, AI tool-call, tool result, final AI answer),
# and the REAL tool output (POL-4.2/POL-9.1, not scripted text) appears in the tool message.
for m in _wiring_result["messages"]:
    print(type(m).__name__, "-", getattr(m, "content", None) or getattr(m, "tool_calls", None))

tool_message = _wiring_result["messages"][2]
assert "POL-4.2" in tool_message.content or "POL-9.1" in tool_message.content, (
    "lc_kb_search's real output never reached the tool message — graph wiring is broken."
)
print("\nOK: lc_kb_search was actually invoked by the graph, and its real output flowed "
      "through to the next turn.")


HumanMessage - does my policy cover a rental car?
AIMessage - [{'name': 'lc_kb_search', 'args': {'query': 'rental car'}, 'id': 'call_1', 'type': 'tool_call'}]
ToolMessage - [POL-4.2] Comprehensive coverage does not include rental vehicle reimbursement unless Rider R-12 (Rental Reimbursement) has been purchased separately.
[POL-4.3] Collision coverage applies to damages resulting from an accident involving the insured vehicle and does not extend to third-party rental vehicles.
[POL-9.1] Rider R-12 provides up to INR 1,500/day for rental vehicle costs, capped at 30 days, while the insured vehicle is under repair due to a covered claim.
AIMessage - Rental cars are addressed in the retrieved clauses above.

OK: lc_kb_search was actually invoked by the graph, and its real output flowed through to the next turn.


---

## Appendix — the same resolution-agent pattern in the OpenAI SDK and Gemini SDK

The LangChain appendix above swapped the *orchestration framework* but let `create_agent` run the
perceive → reason → act → observe loop from the Architecture section for you. This section swaps
the *model provider* and, deliberately, does **not** hide the loop — every step from the Architecture
diagram (call the model → check if it asked for a tool → run the tool → send the result back → get
the final answer) is written out by hand, once for OpenAI and once for Gemini. Seeing it manually is
what makes it obvious what `ClaudeSDKClient` and `create_agent` were actually doing for you above.

Optional, same as the LangChain section — framework/model literacy, not required for anything later
in the week. `pip install openai google-genai`.

**Two new environment variables, easy to confuse with `ANTHROPIC_API_KEY`:**
- OpenAI: `export OPENAI_API_KEY=your-key`
- Gemini: `export GEMINI_API_KEY=your-key` (the SDK also accepts `GOOGLE_API_KEY`; if both are set,
  `GOOGLE_API_KEY` wins — pick one and stick to it)

**What's reused, verbatim, from Step 1:** `policy_chunks` and `search()` — same knowledge base, same
retrieval logic. `grounded_options.system_prompt` from Step 4 is reused too, as the grounding
instruction handed to both new models — same mandate ("you MUST call kb_search first... always cite
the clause id... never fill gaps from general knowledge"), because grounding is a prompt discipline,
not a Claude-specific one. Only the plumbing that gets the model from *decided to call a tool* to
*has the tool's answer* changes below.

In [18]:
# One tool implementation, shared by both SDKs below — same search() and policy_chunks
# from Step 1, just wrapped as a plain function instead of a claude_agent_sdk @tool or a
# langchain @lc_tool. Every framework in this notebook is calling the SAME retrieval logic;
# only the calling convention around it changes.
def kb_search_impl(query: str) -> str:
    results = search(query)
    return "\n".join(f"[{r['id']}] {r['text']}" for r in results)

KB_SEARCH_DESCRIPTION = (
    "Search the insurance policy knowledge base for clauses relevant to a customer's question. "
    "Always call this before answering any question about what is or isn't covered."
)

print("kb_search_impl defined — shared by the OpenAI and Gemini sections below.")


kb_search_impl defined — shared by the OpenAI and Gemini sections below.


### OpenAI SDK — Chat Completions, manual tool loop

**Docs:** https://platform.openai.com/docs/guides/function-calling

The tool is declared as a JSON schema dict (`openai_tools` below) — no decorator, because the plain
Chat Completions API doesn't have one; the schema is data you hand to the API call, same shape every
time. The loop is two round-trips:

1. Send the conversation + `tools=openai_tools`. If the model wants to call `kb_search`, the reply's
   `message.tool_calls` is populated instead of `message.content`.
2. Run the tool yourself, append **both** the assistant's tool-call message and a `role="tool"`
   result message (matched by `tool_call_id`), and call the API again for the final answer.

**Two gotchas that are easy to lose an hour to:**
- `tool_call.function.arguments` is a **JSON string**, not a dict — skip `json.loads(...)` and you'll
  pass a string where `kb_search_impl` expects a keyword argument, and get a confusing `TypeError`
  several lines away from the actual cause.
- The follow-up call fails with a 400 if either the assistant tool-call message or the matching
  `role="tool"` message is missing from `messages` — the API expects to see the full round trip, not
  just the final result.

▶ **Run this** (needs a real `OPENAI_API_KEY`) — expect the same correctly-cited, grounded answer as
Step 4, from a different model and a hand-written loop instead of the SDK's or LangChain's.

In [19]:
import json
from openai import OpenAI

openai_client = OpenAI()   # reads OPENAI_API_KEY from the environment

openai_tools = [{
    "type": "function",
    "function": {
        "name": "kb_search",
        "description": KB_SEARCH_DESCRIPTION,
        "parameters": {
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    },
}]

question = "My car's in the shop after an accident — does my policy cover a rental car?"
messages = [
    {"role": "system", "content": grounded_options.system_prompt},   # same mandate as Step 4
    {"role": "user", "content": question},
]

# Swap "gpt-5-mini" for whatever model id is live on your account if this 404s —
# the loop below is what matters, not the exact model string.
response = openai_client.chat.completions.create(
    model="gpt-5-mini", messages=messages, tools=openai_tools,
)
message = response.choices[0].message

if message.tool_calls:
    messages.append(message)   # the assistant's tool-call turn — required before the tool reply
    for tool_call in message.tool_calls:
        args = json.loads(tool_call.function.arguments)   # a JSON STRING, not a dict — must parse
        result_text = kb_search_impl(**args)
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,   # must match the id the model issued, or the API 400s
            "content": result_text,
        })
    final = openai_client.chat.completions.create(
        model="gpt-5-mini", messages=messages, tools=openai_tools,
    )
    print(final.choices[0].message.content)
else:
    # Model answered without calling the tool — same ungrounded-answer risk as Step 3,
    # worth noticing if it happens here too.
    print(message.content)


OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

### A faster, offline check on the OpenAI wiring

Same idea as Lab 2 and Lab 3's offline checks: no API call, no `OPENAI_API_KEY` needed. This proves
`kb_search_impl` — the function the loop above dispatches to once it parses `tool_call.function.arguments`
— returns the right, citable clauses. It does **not** prove the model will decide to call the tool;
that needs a real key and the cell above.

In [ ]:
# Offline check — no API call needed. Simulates the JSON-string arguments the OpenAI API
# actually sends (see the gotcha above), to prove the parse-then-dispatch step is correct
# independent of whether a real model call ever happens.
def _test_openai_dispatch():
    fake_arguments_json = json.dumps({"query": "does my policy cover a rental car after an accident"})
    args = json.loads(fake_arguments_json)
    result_text = kb_search_impl(**args)
    assert "POL-4.2" in result_text and "POL-9.1" in result_text, (
        "kb_search_impl did not return the expected clauses — check search() from Step 1."
    )
    print(result_text)
    print("\nOK: kb_search_impl dispatches correctly from parsed OpenAI-style tool arguments.")

_test_openai_dispatch()


### Gemini SDK — manual function calling

**Docs:** https://ai.google.dev/gemini-api/docs/function-calling

Structurally the same loop as the OpenAI section — declare the tool, call the model, check whether
it asked for a tool, run it, send the result back — with the details in different places:

- The tool is a `types.FunctionDeclaration` wrapped in a `types.Tool`, not a raw JSON dict.
- The grounding instruction goes in `GenerateContentConfig(system_instruction=...)`, not a
  `{"role": "system", ...}` message in the conversation list.
- **The opposite gotcha from OpenAI:** `function_call.args` here is **already a dict** — passing it
  straight to `kb_search_impl(**fc.args)` just works. Muscle memory from the OpenAI section
  (`json.loads(...)` first) will crash this one; there's nothing to parse.
- Building the second turn means constructing `types.Content` objects by hand for the whole exchange
  so far (user question → model's function call → the function's result) — more verbose than
  OpenAI's flat `messages` list, but the same three-message shape underneath.

**Worth knowing, not used here:** the Gemini SDK also supports *automatic* function calling — pass a
plain Python function straight into `tools=[...]` and the SDK handles the call-and-continue loop for
you, closer to how `create_agent` behaves in the LangChain section. It's skipped on purpose so the
mechanics stay visible, the same reason the OpenAI section above is manual too.

▶ **Run this** (needs a real `GEMINI_API_KEY`) — expect the same grounded, cited answer, from a third
model and a third calling convention, over the identical knowledge base.

In [ ]:
from google import genai
from google.genai import types

gemini_client = genai.Client()   # reads GEMINI_API_KEY (or GOOGLE_API_KEY) from the environment

kb_search_declaration = types.FunctionDeclaration(
    name="kb_search",
    description=KB_SEARCH_DESCRIPTION,
    parameters={
        "type": "object",
        "properties": {"query": {"type": "string"}},
        "required": ["query"],
    },
)
kb_search_tool = types.Tool(function_declarations=[kb_search_declaration])

# Swap "gemini-3-flash" for whatever model id is live on your account if this 404s —
# the loop below is what matters, not the exact model string.
GEMINI_MODEL = "gemini-3-flash"
gemini_config = types.GenerateContentConfig(
    system_instruction=grounded_options.system_prompt,   # same mandate as Step 4, again
    tools=[kb_search_tool],
)

response = gemini_client.models.generate_content(
    model=GEMINI_MODEL, contents=question, config=gemini_config,   # `question` from the OpenAI cell
)

function_call_part = next(
    (p for p in response.candidates[0].content.parts if p.function_call), None
)

if function_call_part:
    fc = function_call_part.function_call
    result_text = kb_search_impl(**fc.args)   # fc.args is ALREADY a dict — no json.loads here

    follow_up = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=[
            types.Content(role="user", parts=[types.Part(text=question)]),
            types.Content(role="model", parts=[function_call_part]),
            types.Content(role="user", parts=[
                types.Part.from_function_response(name=fc.name, response={"result": result_text})
            ]),
        ],
        config=gemini_config,
    )
    print(follow_up.text)
else:
    # Model answered without calling the tool — same ungrounded-answer risk as Step 3.
    print(response.text)


### A faster, offline check on the Gemini wiring

Same purpose as the OpenAI offline check: no API call, no `GEMINI_API_KEY` needed. It builds a
synthetic `function_call` `Part` by hand — the shape the real API would return if the model decided
to call `kb_search` — and proves both that `kb_search_impl` dispatches correctly from Gemini's
already-a-dict `args`, and that `Part.from_function_response` builds without error.

In [ ]:
# Offline check — no API call needed. Constructs the same Part shape the real Gemini
# response carries a function call in, without ever hitting the network.
def _test_gemini_wiring():
    fake_part = types.Part(function_call=types.FunctionCall(
        name="kb_search",
        args={"query": "does my policy cover a rental car after an accident"},   # already a dict
    ))
    fc = fake_part.function_call
    result_text = kb_search_impl(**fc.args)
    assert "POL-4.2" in result_text and "POL-9.1" in result_text, (
        "kb_search_impl did not return the expected clauses — check search() from Step 1."
    )

    # Round-trip the result back into a function_response Part, same as the live cell does —
    # proves the SDK objects wire together correctly, independent of any model call.
    response_part = types.Part.from_function_response(name=fc.name, response={"result": result_text})
    assert response_part.function_response.name == "kb_search"

    print(result_text)
    print("\nOK: kb_search_impl dispatches correctly from Gemini-style (already-dict) function "
          "args, and Part.from_function_response builds correctly.")

_test_gemini_wiring()


**What's identical across all three appendix sections (LangChain, OpenAI, Gemini), and what's
different:**

| | Tool declaration | Grounding instruction | Model decides to call a tool | You execute it | Result goes back as |
|---|---|---|---|---|---|
| LangChain | `@lc_tool` reads a docstring + type hints | `system_prompt=` on `create_agent` | hidden inside `create_agent`'s loop | hidden | hidden |
| OpenAI | JSON schema dict | `{"role": "system", ...}` message | `message.tool_calls`, args as a **JSON string** | you, by hand | a `role="tool"` message, matched by `tool_call_id` |
| Gemini | `types.FunctionDeclaration` | `GenerateContentConfig(system_instruction=...)` | a `function_call` `Part`, args as an **already-parsed dict** | you, by hand | a `types.Content` built from `Part.from_function_response` |

**The takeaway, stated plainly, same as the LangChain section:** three different SDKs, three
different object shapes, one identical loop — the perceive → reason → act → observe loop from the
Architecture section at the top of this notebook. Grounding survives the swap every time because
it's a property of the system prompt, not of any one vendor's tool-calling implementation. When a
grounded answer comes back wrong from any framework, the fix is the same first move regardless of
which SDK is in front of you: check whether the tool actually got called, then check whether the
system prompt still says it *must* be.